# 🗄️ Project Backend: Microsoft Azure & PostgreSQL Implementation
**Project:** EntiLytics  
**Cloud Provider:** Microsoft Azure for Students

This notebook serves as the formal technical documentation for the EntiLytics database. It covers the initial cloud provisioning via Azure CLI, the relational schema design, and the programmatic connection using SQLAlchemy.

## ☁️ Phase 1: Infrastructure Provisioning
The following steps were performed using the **Azure CLI** to establish the cloud environment. This ensures the backend is hosted on a persistent, managed service rather than a local machine.

### **Deployment Steps:**
1. **Service Registration:** Enabled the `Microsoft.DBforPostgreSQL` namespace for the subscription.
2. **Resource Management:** Created `entilytics-rg` to house all project assets.
3. **Server Deployment:** Provisioned a **Flexible Server** (Burstable B1ms) in the `japanwest` region.
4. **Network Security:** Configured firewall rules (`0.0.0.0`) to allow remote connection from the development environment.

In [ ]:
# The following commands were executed in the terminal to initialize the environment.
# These are kept as a reference for reproducibility.

# Register PostgreSQL Provider
az provider register --namespace Microsoft.DBforPostgreSQL

# Create Resource Group
az group create --name entilytics-rg --location japanwest

# Create the Flexible Server
az postgres flexible-server create \
    --resource-group entilytics-rg \
    --name entilytics-db-server \
    --location japanwest \
    --admin-user enti_admin \
    --admin-password <SECURE_PASSWORD> \
    --sku-name Standard_B1ms \
    --tier Burstable \
    --public-access 0.0.0.0

# Create the logical database
az postgres flexible-server db create \
    --resource-group entilytics-rg \
    --server-name entilytics-db-server \
    --database-name entilytics_db

## 🔐 Phase 2: Python Environment & Security
To maintain security, database credentials are not hardcoded. We utilize a `.env` file and the `python-dotenv` library to inject credentials into the application context.

In [ ]:
# Load .env from the root directory
load_dotenv()

# Database Connection
DATABASE_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}@{os.getenv('DB_HOST')}:5432/{os.getenv('DB_NAME')}?sslmode=require"

engine = create_engine(DATABASE_URL)
SessionLocal = sessionmaker(bind=engine)

🛠️ Data Definition Language (SQL Schema)

The following SQL script represents the physical layer of the database. Even though the tables are already provisioned, this cell serves as the official technical record of the schema, including relationships and constraints.

In [ ]:
def verify_schema():
    schema = """
    -- Reference Tables --
    CREATE TABLE EntityType (
        EntityTypeID BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        -- GENERATED ALWAYS AS IDENTITY = modern PostgreSQL standard
        TypeName VARCHAR(100) NOT NULL
    );
    CREATE TABLE Source (
        SourceID BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        Name VARCHAR(255),
        URL TEXT
    );
    -- User Management --
    CREATE TABLE Account (
        AccountID BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        Gmail VARCHAR(100) UNIQUE NOT NULL,
        Account_Role VARCHAR(255) NOT NULL DEFAULT 'user',
        created_at TIMESTAMPTZ DEFAULT NOW()
    );
    CREATE INDEX idx_account_gmail ON Account(Gmail);
    -- Content --
    CREATE TABLE Entity (
        EntityID BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        Name VARCHAR(255) NOT NULL,
        EntityTypeID BIGINT REFERENCES EntityType(EntityTypeID)
    );
    CREATE TABLE Article (
        ArticleID BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        Title TEXT,
        Content TEXT,
        SourceID BIGINT REFERENCES Source(SourceID),
        DatePublished DATE,
        created_at TIMESTAMPTZ DEFAULT NOW()
    );
    -- NLP Results --
    CREATE TABLE EntityExtraction (
        EntityExtractionID BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        ArticleID BIGINT REFERENCES Article(ArticleID) ON DELETE CASCADE,
        -- ON DELETE CASCADE = deleting an Article from Admin page removes related summaries, rankings, and notes automatically.
        EntityID BIGINT REFERENCES Entity(EntityID),
        Position INT,
        Frequency INT
    );
    CREATE TABLE EntityImportance (
        ImportanceID BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        ExtractionID BIGINT REFERENCES EntityExtraction(EntityExtractionID) ON DELETE CASCADE,
        ImportanceScore FLOAT
    );
    CREATE TABLE RelationshipMap (
        RelationshipID BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        ArticleID BIGINT REFERENCES Article(ArticleID) ON DELETE CASCADE,
        EntityA_ID BIGINT REFERENCES Entity(EntityID),
        EntityB_ID BIGINT REFERENCES Entity(EntityID)
    );
    -- User-specific Data --
    CREATE TABLE UserArticle (
        UserArticleID BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        AccountID BIGINT REFERENCES Account(AccountID),
        ArticleID BIGINT REFERENCES Article(ArticleID) ON DELETE CASCADE,
        DateStored TIMESTAMPTZ DEFAULT NOW()
    );
    CREATE TABLE Annotation (
        AnnotationID BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        AccountID BIGINT REFERENCES Account(AccountID),
        ArticleID BIGINT REFERENCES Article(ArticleID) ON DELETE CASCADE,
        Note TEXT
    );
    CREATE TABLE Summary (
        SummaryID BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        AccountID BIGINT REFERENCES Account(AccountID),
        ArticleID BIGINT REFERENCES Article(ArticleID) ON DELETE CASCADE,
        SummaryText TEXT
    );
    """
    with engine.connect() as conn:
        conn.execute(text(schema))
        conn.commit()
    print("SQL Schema verified and synchronized with Azure.")

verify_schema()

🐍 SQLAlchemy ORM Models

To prevent ImportError in the main app.py, we define the Python classes that map to the SQL tables above. These allow the application to perform CRUD operations (Create, Read, Update, Delete) using Python objects instead of raw SQL strings.